In [1]:
from tgrocery import Grocery
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
pd.set_option("display.precision", 4)

In [2]:
dt = pd.read_csv('data/final_s_20220823.tsv', encoding='utf-8', delimiter='\t', names=['txt','label'])
dt.head(2)

,txt,label
0,000100资金流向,金融服务|投资理财
1,000157资金流向,金融服务|投资理财


In [3]:
dt[dt['label'].str.startswith('游戏')]['label'].value_counts()

游戏|单机游戏    4327
游戏|其他      3528
游戏|网页游戏    2224
游戏|客户端     1533
游戏|手游      1444
游戏|游戏服务     915
Name: label, dtype: int64

In [4]:
mapping = pd.read_excel('data/query和兴趣标注体系对应.xlsx', sheet_name='query体系')
mapping.head(2)

,一级分类,二级分类,三级,Unnamed: 3,兴趣一级,兴趣二级,兴趣三级
0,IT产品,IT产品|其他,NaN,NaN,IT产品,其他,NaN
1,IT产品,IT产品|外设,NaN,NaN,IT产品,其他,NaN


In [5]:
mapping['interest'] = mapping['兴趣一级']
mapping.loc[mapping['兴趣二级'].isnull()==False,'interest'] = mapping.loc[mapping['兴趣二级'].isnull()==False,'interest'] +'|' + \
    mapping.loc[mapping['兴趣二级'].isnull()==False,'兴趣二级'] 
mapping.loc[mapping['兴趣三级'].isnull()==False,'interest'] = mapping.loc[mapping['兴趣三级'].isnull()==False,'interest'] +'|' + \
    mapping.loc[mapping['兴趣三级'].isnull()==False,'兴趣三级'] 

In [6]:
mapping['query'] = mapping['二级分类']
mapping.loc[mapping['三级'].isnull()==False,'query'] = mapping.loc[mapping['三级'].isnull()==False,'三级']
mapping['query'].nunique()

313

In [7]:
mapping = mapping[mapping['interest'].isnull()==False]

In [8]:
mapping['query'] = mapping['query'].str.replace('餐饮美食','食品美食')

In [9]:
query_map = {}
for row in mapping.loc[mapping['interest'].isnull()==False,['query','interest']].values:
    query_map[row[0]] = row[1]

In [10]:
query_map['人名'] = '人名'

In [11]:
dt.loc[dt['label'].isin(mapping['query'])==False, 'label'].unique()

array(['人名'], dtype=object)

In [12]:
dt = dt[dt['label'].isin(query_map)]
dt.shape

(1041617, 2)

In [13]:
dt['label'].nunique()

313

In [14]:
dt['label'] = dt['label'].map(lambda x: query_map[x])
dt['label'].nunique()

120

In [15]:
dt['label'].value_counts().tail()

医疗健康|医疗服务|养老院    26
旅游|交通票务|车船票务     25
家电爱好者|个护电器       23
图书阅读|经管          21
体育运动|运动鞋服        12
Name: label, dtype: int64

In [16]:
rule = pd.read_csv('data/rule_dict_interest.txt', sep='\t', names=['txt','label'])
rule.head(2)

,txt,label
0,绝对军事,娱乐休闲|综艺
1,巨兽来袭2,娱乐休闲|电影


In [17]:
def get_interest(x):
    if query_map.get(x):
        return query_map[x]
    else:
        return x

In [18]:
rule_label = rule['label'].unique()
for i in rule_label:
    if i not in query_map.values():
        print(i)

娱乐休闲|综艺
娱乐休闲|电影
娱乐休闲|记录片
娱乐休闲|电视剧
娱乐休闲|动漫
游戏|休闲益智
游戏|动作冒险
游戏|战争策略
游戏|角色扮演


In [19]:
rule['label'] = rule['label'].map(lambda x: get_interest(x))

In [20]:
dt[dt['label'].str.startswith('图书')]['label'].value_counts()

图书阅读|小说      6051
图书阅读|其他      3503
图书阅读|报纸杂志    1901
图书阅读|教育      1737
图书阅读|动漫       131
图书阅读|生活        47
图书阅读|科技        28
图书阅读|经管        21
Name: label, dtype: int64

In [21]:
rule['label'].unique()

array(['娱乐休闲|影视|综艺', '娱乐休闲|影视|电影', '娱乐休闲|影视|记录片', '娱乐休闲|影视|电视剧',
       '娱乐休闲|影视|动漫', '图书阅读|动漫', '图书阅读|小说', '图书阅读|教育', '图书阅读|生活',
       '图书阅读|科技', '图书阅读|经管', '游戏|休闲益智', '游戏|其他', '游戏|动作冒险', '游戏|战争策略',
       '游戏|角色扮演'], dtype=object)

In [22]:
dt.shape

(1041617, 2)

In [23]:
# dt = dt[(dt['txt'].isin(rule['txt'])&(dt['label'].str.contains('游戏')))==False]
# dt.shape

In [24]:
# dt.loc[(dt['txt'].isin(rule['txt']))&(dt['label'].str.contains('游戏')==False)].shape

In [25]:
# dt[dt['label'].str.contains('游戏')].head()
dt = dt[dt['label']!='游戏']
dt.shape

(1031174, 2)

In [26]:
# rule = rule[rule['txt'].isin(dt['txt'])==False]

In [27]:
rule.shape

(1422698, 2)

In [28]:
# rule2 = rule.copy()

In [29]:
import re
def get_chinese_ratio(x):
    pattern = re.compile(r'[^\u4e00-\u9fa5]')
    chinese = re.sub(pattern, '', x)
    return(len(chinese)/len(x))

In [30]:
# rule['txt'] = rule['txt'].astype(str)
# rule2['chinese_ratio'] = rule['txt'].apply(lambda x: get_chinese_ratio(x))
# rule2 = rule2[rule2['chinese_ratio']>0.5]

In [31]:
# np.random.seed(1)
# add_list = []
# for i in rule2.loc[rule2['label'].str.startswith('娱乐休闲'),'label'].unique():
#     add_list.append(rule2.loc[rule2['label']==i].sample(min(1000, np.sum(rule2['label']==i))))
# for i in rule2.loc[rule2['label'].str.startswith('图书'),'label'].unique():
#     add_list.append(rule2.loc[rule2['label']==i].sample(min(500, np.sum(rule2['label']==i))))
# for i in rule2.loc[rule2['label'].str.startswith('游戏'),'label'].unique():
#     add_list.append(rule2.loc[rule2['label']==i].sample(min(2000, np.sum(rule2['label']==i))))

In [32]:
# add_list = pd.concat(add_list)
# add_list.shape

In [16]:
# add_list[['txt','label']].to_csv('data/additional_interest_train.txt', sep='\t', index=False, header=None)
add_list = pd.read_csv('data/additional_interest_train.txt', encoding='utf-8', delimiter='\t', names=['txt','label'])
game_other = add_list[(add_list['label']=='游戏|其他')&(add_list.index>16000)]
add_list = add_list[(add_list['label']!='游戏|其他')]
game_other.head(2)

,txt,label
16279,SH01-幻炎（永久）虹光水晶*40绿钻石*40 礼包已领取,游戏|其他
16283,Switchlite开机,游戏|其他


In [17]:
add_list = pd.concat([add_list, game_other])

In [18]:
add_list2 = []
for i in range(100):
    add_list2.append([str(i), '其他'])
add_list2 = pd.DataFrame(add_list2, columns=['txt','label'])

In [19]:
add_list2.head(2)

,txt,label
0,0,其他
1,1,其他


In [20]:
add_list.loc[add_list['label'].isin(query_map),'label']= add_list.loc[add_list['label'].isin(query_map),'label'].map(lambda x: query_map[x])

In [21]:
add_list['label'].value_counts()

游戏|角色扮演        2005
游戏|动作冒险        1997
游戏|休闲益智        1891
游戏|战争策略        1865
娱乐休闲|影视|电视剧     998
娱乐休闲|影视|电影      998
娱乐休闲|影视|动漫      975
娱乐休闲|影视|综艺      932
娱乐休闲|影视|记录片     583
图书阅读|小说         497
图书阅读|经管         474
图书阅读|教育         330
图书阅读|动漫         316
图书阅读|科技         282
图书阅读|生活         149
其他               31
游戏|其他             7
软件应用|系统工具         5
化工及材料             2
软件应用|办公学习         2
娱乐休闲|影视           1
成人|成人             1
教育培训|职业教育         1
办公用品              1
旅游|其他             1
交通类               1
娱乐休闲|音乐           1
Name: label, dtype: int64

In [22]:
dt = pd.concat([dt, add_list, add_list2])

In [23]:
dt = dt[dt['txt'].duplicated(keep='first')==False]

In [24]:
dt['label'].value_counts()

机械设备             134184
医疗健康|疾病          128341
电子电工              65492
农林牧渔              57805
商务服务              53974
                  ...  
体育运动|羽毛球             30
医疗健康|医疗服务|养老院        26
旅游|交通票务|车船票务         25
家电爱好者|个护电器           23
体育运动|运动鞋服            12
Name: label, Length: 124, dtype: int64

In [25]:
dt = dt.reset_index(drop=True)
dt.shape

(1049628, 2)

In [26]:
dt.to_csv('data/train_interest_220829.tsv', sep='\t', index=False, header=None)

In [44]:
train_src = []
for i, j in zip(dt['txt'].values, dt['label'].values):
    train_src.append((j, i))
train_src[:5]

[('金融|投资理财', '000100资金流向'),
 ('金融|投资理财', '000157资金流向'),
 ('金融|投资理财', '000709资金流向'),
 ('交通类', '000889渤海物流'),
 ('金融|投资理财', '000920股吧东方财富网股吧')]

In [36]:
X_train, X_test, y_train, y_test = train_test_split(dt['txt'].values, dt['label'].values, test_size=0.3, random_state=42)

In [37]:
X_train.shape,len(y_train)

((523497,), 523497)

In [38]:
train_src = []
for i, j in zip(X_train, y_train):
    train_src.append((j,i))
train_src[:2]

[('商务服务', '招聘女主持'), ('商务服务', '长沙展览服务有限公司')]

In [39]:
test_src = []
for i, j in zip(X_test, y_test):
    test_src.append((j,i))
test_src[:4]

[('医疗健康|疾病', '割痔疮需要多少钱'), ('医疗健康|疾病', '艾力可片'), ('农林牧渔', '野猪种'), ('人名', '木村千咲')]

In [45]:
grocery = Grocery('interest_20220816')
grocery.train(train_src)

Building prefix dict from the default dictionary ...
Dumping model to file cache /tmp/jieba.cache
Loading model cost 1.995 seconds.
Prefix dict has been built successfully.


*.**.**.*
optimization finished, #iter = 32
Objective value = -159759.591997
nSV = 3632842


In [135]:
test_result = grocery.test(test_src)
test_result.accuracy_overall

0.8534567675624117

In [136]:
from sklearn.metrics import classification_report, accuracy_score

In [137]:
accuracy_score(np.array(test_result.true_y)[y_test!='人名'], np.array(test_result.predicted_y)[y_test!='人名'])

0.8547311510334148

In [138]:
report = classification_report(test_result.true_y, test_result.predicted_y, output_dict=True)
df = pd.DataFrame(report).transpose()
df.head(2)

/home/hdp-portrait/.local/lib/python3.7/site-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hdp-portrait/.local/lib/python3.7/site-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hdp-portrait/.local/lib/python3.7/site-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


,precision,recall,f1-score,support
IT产品,0.9112,0.8759,0.8932,3045.0
IT产品爱好者|手机数码,0.6884,0.5649,0.6205,262.0


In [139]:
df[(df['f1-score']<0.5) & (df['support']>100)].sort_values('support', ascending=False)

,precision,recall,f1-score,support
医疗健康,0.7213,0.3774,0.4956,1330.0
金融服务,0.6536,0.3468,0.4532,1012.0
图书阅读|小说,0.4460,0.3808,0.4108,759.0
图书阅读|教育,0.3769,0.2996,0.3338,751.0
游戏|休闲益智,0.4740,0.2799,0.3520,618.0
游戏|动作冒险,0.3468,0.2409,0.2843,606.0
游戏|战争策略,0.4793,0.4649,0.4720,598.0
游戏|角色扮演,0.4218,0.3797,0.3996,582.0
娱乐休闲|影视|电影,0.2468,0.1370,0.1762,416.0
娱乐休闲|影视|动漫,0.4181,0.3550,0.3840,338.0


In [74]:
np.mean((df['f1-score']<0.5) & (df['support']>100))

0.09523809523809523

In [75]:
test_result = []
for i,j in zip(X_test, y_test):
    pred = grocery.predict(i)
    pred_label = pred.predicted_y
    score=max(pred.dec_values.values())
    test_result.append([i,j,pred_label,score])

In [76]:
test_result = pd.DataFrame(test_result, columns = ['txt', 'cate', 'pred', 'score'])
test_result.head()

,txt,cate,pred,score
0,润天股份,金融服务|投资理财,金融服务|投资理财,1.2967
1,捕鼠,生活服务|虫害控制,生活服务|虫害控制,1.5501
2,大理酒店,旅游|宾馆酒店,旅游|宾馆酒店,1.8960
3,宿舍家具采购,办公用品|商用家具,办公用品|商用家具,1.4897
4,哈高科蛋黄卵磷脂片,医疗健康|药品,医疗健康|药品,1.0088


In [77]:
test_result['cate_1st'] = test_result['cate'].apply(lambda x:x.split('|')[0])
test_result['pred_1st'] = test_result['pred'].apply(lambda x:x.split('|')[0])
test_result.head(2)

,txt,cate,pred,score,cate_1st,pred_1st
0,润天股份,金融服务|投资理财,金融服务|投资理财,1.2967,金融服务,金融服务
1,捕鼠,生活服务|虫害控制,生活服务|虫害控制,1.5501,生活服务,生活服务


In [78]:
report = classification_report(test_result['cate_1st'], test_result['pred_1st'], output_dict=True)
df = pd.DataFrame(report).transpose()
df.head(2)

,precision,recall,f1-score,support
IT产品,0.9027,0.8747,0.8885,3192.0
交通类,0.8204,0.8213,0.8209,4801.0


In [79]:
np.mean(test_result['cate_1st']==test_result['pred_1st'])

0.8984292048504623

In [80]:
ret = []
for th in np.arange(0.4,1,0.1):
    tmp = test_result[test_result['score']>=th]
    acc_1st = np.mean(tmp['cate_1st']==tmp['pred_1st'])
    acc_2nd = np.mean(tmp['cate']==tmp['pred'])
    ret.append([th, acc_1st, acc_2nd, len(tmp)/len(test_result)])
ret = pd.DataFrame(ret, columns = ['阈值','1级类别准确度','标签准确度','标注比例'])

In [81]:
tmp = test_result[test_result['score']>=0.4]

In [82]:
report = classification_report(tmp['cate_1st'], tmp['pred_1st'], output_dict=True)
df = pd.DataFrame(report).transpose()
df.head(2)

,precision,recall,f1-score,support
IT产品,0.9227,0.9322,0.9274,2934.0
交通类,0.8747,0.8735,0.8741,4317.0


In [34]:
tmp.to_excel('data/判断错误2_20220714.xlsx',index=False)

In [83]:
# tmp[tmp['cate_1st']=='工具'][['cate_1st','pred_1st']].value_counts().head(10)

In [85]:
df[(df['f1-score']<0.8)].sort_values('support', ascending=False)

,precision,recall,f1-score,support
资讯,0.6401,0.6093,0.6243,3041.0
其他,0.7619,0.7476,0.7547,2393.0
工具,0.6995,0.6903,0.6949,1737.0
图书阅读,0.8194,0.7808,0.7997,1209.0
体育运动,0.7997,0.6837,0.7371,765.0
电子商务,0.7195,0.6344,0.6743,558.0
人名,0.6714,0.6257,0.6477,529.0
办公用品,0.8426,0.7034,0.7667,236.0
时尚艺术,0.6456,0.2849,0.3953,179.0
家用电器,0.6705,0.7073,0.6884,164.0


In [56]:
# tmp[(tmp['cate_1st']=='图书阅读') & (tmp['pred_1st']!='图书阅读')].sample(20)

In [46]:
grocery2 = Grocery('UserInterest_0924')
grocery2.load()

In [52]:
# hunxiao = ['123百度','1百度176','百度一+下','百度456','百  度','百度163']
# test = pd.read_csv('data/sample_freq_query_20220809.tsv', names=['query'])
test = pd.read_csv('data/sample_freq_query_20220822.tsv', names=['query'])
# test = pd.read_csv('data/sample_query_10k_20220809.tsv', names=['query'])
test.head(2)

,query
0,贵州省专业技术人员继续教育平台
1,杭州


In [53]:
query_map = {}
for row in rule.values:
    query_map[row[0]] = row[1]

In [54]:
remove_list = ['免费','观看','全集','完整版','在线','下载','全文','阅读']
def simplify_ys(txt):
    for i in remove_list:
        txt = txt.replace(i,'')
    return txt

In [55]:
strong_rules = {'电影在线观看':'娱乐休闲|影视|电影','电视剧免费观看':'娱乐休闲|影视|电视剧','韩剧在线观看':'娱乐休闲|影视|电视剧',
                '全文阅读':'图书阅读|小说','全文免费阅读':'图书阅读|小说',
               '动漫在线观看':'娱乐休闲|影视|动漫','漫画全集免费':'娱乐休闲|影视|动漫'}

In [56]:
query_map['长歌行']='娱乐休闲|影视|电视剧'

In [57]:
test['query'] = test['query'].astype(str)

In [58]:
result = []
for i in test['query'].values:
    predicted = False
    for key, val in strong_rules.items():
        if key in i:
            pred_label = strong_rules.get(key)
            score = 'rule'
            predicted=True
    if not predicted:
        i_simp = simplify_ys(i)
        if query_map.get(i_simp) is None:
            pred = grocery.predict(i)
            pred_label = pred.predicted_y
            score=max(pred.dec_values.values())
            if score < 0.45:
                score=''
        else:
            pred_label = query_map.get(i_simp)
            score = 'match'
    pred2 = grocery2.predict(i)
    pred_label2 = pred2.predicted_y
    score2=max(pred2.dec_values.values())
    result.append([i,pred_label,score,pred_label2,score2])

In [59]:
result = pd.DataFrame(result, columns = ['query', '新标签', '新score', '旧标签','旧score'])
result.head(2)

,query,新标签,新score,旧标签,旧score
0,贵州省专业技术人员继续教育平台,教育培训|职业教育,1.6172,教育|职业教育,1.3436
1,杭州,旅游|其他,0.5439,旅游|住宿,0.5298


In [60]:
result.shape

(500, 5)

In [61]:
result = result[result['新score']!='']
result.shape

(451, 5)

In [67]:
# writer = pd.ExcelWriter('data/query_interest_按类别验证_20220815.xlsx', engine='xlsxwriter')
# for i in result['新标签'].unique():
#     if np.sum(result['新标签']==i) > 0:
#         out = result.loc[result['新标签']==i].sample(min(50, np.sum(result['新标签']==i)))
#         out = out.sort_values(['旧标签','query']).to_excel(
#             writer, index=False, sheet_name = i)
# writer.save()

In [64]:
# result = result[result['新标签']!=result['旧标签']]
result.sort_values(['新标签','旧标签','query']).to_excel('data/interest_新旧模型对比_20220824.xlsx', index=False)

In [61]:
np.random.seed(22)
result = result.sample(frac=1)
result = result.groupby('新标签').head(50)
result = result.sort_values(['新标签','旧标签','query'])
# result.drop(columns=['label_1st']).to_excel('data/新旧模型对比_按类别2_20220818.xlsx', index=False)
result.to_excel('data/interest_新旧模型对比_20220823.xlsx', index=False)

In [109]:
ret = pd.read_excel('data/interest_新旧模型对比_20220812.xlsx')
ret.head(2)

,query,新标签,新score,旧标签,旧score,对比结果
0,高清电脑壁纸,IT产品|其他,0.7273,图书音像,0.9600,0.0
1,惠普打印机,IT产品|其他,1.5331,办公文教,2.3438,0.0


In [110]:
ret = ret[['query','新标签','对比结果']]
ret = ret.rename(columns={'新标签':'新标签_old'})
result = pd.merge(result, ret, how='left', on='query')

In [111]:
result.loc[result['新标签']!=result['新标签_old'],'对比结果']=''

In [112]:
np.sum(result['对比结果']=='')

20

In [113]:
result = result.sort_values('新标签')
result.drop(columns=['新标签_old']).to_excel('data/interest_新旧模型对比_20220812.xlsx', index=False)

In [1]:
grocery.save()

NameError: name 'grocery' is not defined

In [4]:
grocery = Grocery('UserInterest')
grocery.load()

In [31]:
val_object = grocery.predict('人工智能')
val_pred_label = val_object.predicted_y
val_pred_label

u'\u56fe\u4e66\u97f3\u50cf|\u957f\u89c6\u9891'

In [26]:
print(u'\u56fe\u4e66\u97f3\u50cf|\u52a8\u6f2b')

In [9]:
max(val_object.dec_values.values())

0.9888343788836396

In [10]:
val_pred_label == u'IT产品'

True

In [44]:
for i, j in zip(top_queries['query'].values, top_queries['hangye'].values):
    val_object = grocery.predict(i)
    if j != val_object.predicted_y:
        print i, j, val_object.predicted_y

 4339网页游戏大全 游戏|网络小游戏 游戏|其他
nba视频直播在线观看 休闲娱乐|运动 视频|视频
中国人事考试网 教育|其他 教育|高考自考
企业微信 IT产品 社交|交友
学信网 教育|高考自考 教育|其他
学科网 教育|K12 教育|其他
快看漫画 休闲娱乐|动漫 视频|视频
有道 教育|语言培训 视频|视频
死神vs火影 休闲娱乐|动漫 视频|视频
火影忍者 休闲娱乐|动漫 视频|视频
英雄联盟 游戏|端游 视频|视频


In [2]:
l = '一件,衣服,真相,有人,把,魔爪,伸向,中国,孩子,有人,把,魔爪,伸向,中国,孩子'
print(l.replace(',', ''))

一件衣服真相有人把魔爪伸向中国孩子有人把魔爪伸向中国孩子
